# 04. Cortex Agent ― Analyst と Search を組み合わせる

**Cortex Agent** は、質問を理解して**どのツールを使うかを自分で判断し**、結果を回答にまとめる AI エージェントです。

| ツール | 中身 | 得意な質問 |
|---|---|---|
| `sales_ops_analyst`（Cortex Analyst） | 02 の Semantic View v2 | 「何件？」「いくら？」「誰？」 |
| `sales_docs_search`（Cortex Search） | 03 の PDF 検索 | 「推奨は？」「どう対応すべき？」「なぜ？」 |

**両方を使うと**、「レポートの推奨値を、実際のデータが満たしているか」のような質問に 1 回で答えられます。

**前提**：`03_cortex_search.ipynb` を実行済みであること  
**所要時間**：約15分

> 🖼️ **図の差し込み位置：Agent の処理の流れ**  
> 質問 → Agent が計画 → Analyst（数字）／Search（文書）を呼ぶ → 結果を統合して回答  
> （画像は `images/` に置き、`![Agent の処理の流れ](../images/xxx.png)` で差し込んでください）

## Step 0. セッションの準備

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")
print("ロール:", session.get_current_role(), "/ ウェアハウス:", session.get_current_warehouse())

## Step 1. Agent を SQL で作る

| spec の項目 | 内容 |
|---|---|
| `models.orchestration` | 使うモデル。`auto` で Snowflake が自動選択 |
| `instructions.orchestration` | **ツールの使い分けの指示**。数字は Analyst、推奨・方針は Search、比較は両方 |
| `instructions.response` | 回答の書き方。日本語の質問には日本語で答える |
| `tools` / `tool_resources` | 2 つのツールと、それぞれの参照先（Semantic View、Search Service） |

> ⚠️ **ハマりどころ①：ウェアハウスの指定位置**
> Analyst の `execution_environment` は spec のトップではなく、**`tool_resources.<ツール名>` の中**に書きます。
> 位置が違うと、作成時に `unrecognized field execution_environment`、実行時に `missing an execution environment` になります。

> ⚠️ **ハマりどころ②：作り直すと権限が消える**
> `CREATE OR REPLACE` は既存の GRANT を引き継ぎません（検証したアカウントでは `CREATE AGENT ... COPY GRANTS` が構文エラーでした）。そのため作成直後に権限を付け直しています。

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT
  COMMENT = 'Sales Operations agent (Cortex Analyst + Cortex Search)'
  FROM SPECIFICATION $$
  models:
    orchestration: auto

  instructions:
    system: >
      You are a Sales Operations analyst. Help users explore revenue, pipeline health,
      sales rep performance, customer segmentation, product mix, and marketing attribution.
    orchestration: >
      Use sales_ops_analyst for numbers from the sales data (revenue, pipeline, reps, customers,
      products, marketing). Use sales_docs_search for recommendations, benchmarks, policies and
      qualitative insights in the internal PDF reports. When a question compares the data with a
      recommendation in the reports, use both tools.
    response: >
      Provide concise, actionable answers. Format USD values as currency and rates as percentages.
      When you use the reports, mention the report name. Always reply in the same language as the
      user's question (reply in Japanese if the question is in Japanese).

  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: sales_ops_analyst
        description: >
          Structured sales operations data: revenue, pipeline, rep performance, customers,
          product plans, and marketing attribution. Includes company KPIs such as stalled deals,
          at-risk deals, win rate and pipeline coverage target.
    - tool_spec:
        type: cortex_search
        name: sales_docs_search
        description: >
          Internal PDF reports (Sales Performance Analysis, Customer Intelligence & Pipeline) with
          benchmarks, recommendations, and qualitative insights.

  tool_resources:
    sales_ops_analyst:
      semantic_view: CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW
      execution_environment:
        type: warehouse
        warehouse: CORTEX_QUICK_HOL_WH
    sales_docs_search:
      name: CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH
      max_results: 4
      title_column: FILE_NAME
      id_column: CHUNK_ID
  $$

In [ ]:
session.sql(f"GRANT USAGE ON AGENT {DB}.{SCHEMA}.SALES_OPS_AGENT TO ROLE {HOL_ROLE}").collect()
print("OK: Agent への USAGE を付与しました")

In [ ]:
%%sql -r show_agent_grants
SHOW GRANTS ON AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT

## Step 2. Agent に質問する

Quick につなぐ前に、Snowflake 上で答えられるかを確認します。**ここで失敗する場合は、Quick からも同じエラーになります。**
各回答の上に「使ったツール」が表示されます。質問によって Agent がツールを使い分けていることを確かめてください。

In [ ]:
import json

def ask_agent(question: str, agent: str = "SALES_OPS_AGENT", show_sql: bool = True):
    """Agent に質問し、回答・使ったツール・生成 SQL・検索した文書を表示する"""
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS R",
                      params=[f"{DB}.{SCHEMA}.{agent}", payload]).collect()[0]["R"]
    resp = json.loads(raw) if isinstance(raw, str) else raw
    texts, sqls, tools, docs = [], [], [], []
    for c in resp.get("content", []):
        if c.get("type") == "text" and c.get("text", "").strip():
            texts.append(c["text"])
        elif c.get("type") == "tool_use":
            tools.append(c.get("tool_use", {}).get("name"))
        elif c.get("type") == "tool_result":
            for x in c.get("tool_result", {}).get("content", []):
                j = x.get("json", {})
                if j.get("sql"):
                    sqls.append(j["sql"])
                for r in j.get("search_results", []) or []:
                    if r.get("doc_title"):
                        docs.append(r["doc_title"])
    print(f"Q: {question}")
    print(f"（使ったツール: {', '.join(t for t in tools if t) or 'なし'}）\n")
    print("\n".join(texts))
    if show_sql:
        for s in sqls:
            print("\n--- Cortex Analyst が生成した SQL ---\n" + s)
    if docs:
        print("\n--- 参照した文書 ---\n" + "\n".join(sorted(set(docs))))
    return resp

| # | 質問 | 使うはずのツール | 期待される結果（目安） |
|---|---|---|---|
| 1 | 地域別の売上を教えて | Analyst | 北米 約 $5.95M が最大 |
| 2 | 停滞商談は何件で、金額はいくら？ | Analyst | 43 件、約 $4.37M |
| 3 | レポートで推奨されているパイプラインカバレッジは？ | Search | 1.6 倍（Sales Performance Analysis） |
| 4 | 推奨カバレッジを下回っている営業担当は何人？推奨値の根拠も教えて | **両方** | 63 人、1.6 倍の根拠 |
| 5 | 停滞商談への推奨アクションは？対象の件数と金額も | **両方** | 役員スポンサーの関与など＋43 件 |

> 💡 1 回の質問には 15〜40 秒ほどかかります。Quick から呼ぶ場合も同じです（60 秒を超えるとタイムアウト）。

In [ ]:
_ = ask_agent("地域別の売上を教えて")

In [ ]:
_ = ask_agent("停滞商談は何件で、金額はいくら？")

In [ ]:
_ = ask_agent("レポートで推奨されているパイプラインカバレッジは？", show_sql=False)

In [ ]:
_ = ask_agent("推奨カバレッジを下回っている営業担当は何人？推奨値の根拠も教えて")

In [ ]:
_ = ask_agent("停滞商談への推奨アクションは？対象の件数と金額も", show_sql=False)

### ハンズオン用ロールで実行できるか確認する

Quick からはハンズオン用ロールで実行されます。ロールを切り替えて同じように答えられるか確かめます。

In [ ]:
session.use_role(HOL_ROLE)
try:
    _ = ask_agent("営業担当の目標達成率トップ3は？", show_sql=False)
    _ = ask_agent("レポートで推奨されているパイプラインカバレッジは？", show_sql=False)
finally:
    session.use_role("ACCOUNTADMIN")

## Step 3. Agent Studio で確認・編集する（GUI）

SQL で作った Agent は、Snowsight の画面からも確認・編集できます。

1. Snowsight 左メニュー **「AI と ML」→「Agents」**（Agent Studio）を開く
2. `SALES_OPS_AGENT` をクリック
3. **「Configuration」** タブで次を確認する
   - **Tools**：`sales_ops_analyst`（Semantic View）と `sales_docs_search`（Cortex Search）の 2 つ
   - **Instructions**：ツールの使い分けの指示
4. 右側のチャット欄で、上の質問を試す。回答の下の **「思考過程（Thinking）」** を開くと、どのツールをどの順に呼んだかが分かります
5. （任意）**Sample questions** に質問例を追加し、右上の **「Save」** をクリック

> ⚠️ GUI で編集した後に、このNotebookの `create_agent` セルを再実行すると、**GUI の変更は上書きされます**。

> 💡 **Snowflake Intelligence でも使えます**：Snowsight 左メニュー **「AI と ML」→「Snowflake Intelligence」** で `SALES_OPS_AGENT` を選ぶと、チャット画面から同じ Agent に質問できます（02 で `use_as_onboarding_question: true` にした質問が候補に出ます）。

> 🖼️ **図の差し込み位置：Agent Studio の Configuration 画面**  
> SALES_OPS_AGENT の Tools に Cortex Analyst と Cortex Search の 2 つが並んでいる画面  
> （画像は `images/` に置き、`![Agent Studio の Configuration 画面](../images/xxx.png)` で差し込んでください）

> ✅ すべての質問に回答が返り、質問 4・5 で 2 つのツールが使われていれば成功です。
>
> ❌ `Access denied for trial accounts`（399504）が出た場合は、トライアルの AI 制限です。01 の「トライアルアカウントの注意」を参照してください。

次は `05_mcp_and_oauth.ipynb` に進みます。